# ALL Met Eyes

In [ ]:
from google.colab import userdata
PAT_XYZ = userdata.get("PAT_XYZ")

In [ ]:
!pip install mediapipe
!pip install ultralytics
!wget https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task
!wget https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/utils.py
!wget https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/utils_paintings.py
!git clone https://{PAT_XYZ}@github.com/acervos-digitais/met-faces-data.git data
!cd data && git config user.name "Thiago Hersan" && git config user.email "thiago.hersan+github@gmail.com"

In [ ]:
import json
import numpy as np
import requests

from os import makedirs, path
from PIL import Image as PImage
from time import sleep, time as timestamp

from utils import get_combined_jsons
from utils_paintings import PaintingsUtils

DATA_DIR = "./data"
JSON_DIR = f"{DATA_DIR}/json"
IMG_DIR = f"{DATA_DIR}/image"

JSON_OBJS_DIR = f"{JSON_DIR}/objects"
JSON_FACES_DIR = f"{JSON_DIR}/faces"
JSON_LANDMARKS_DIR = f"{JSON_DIR}/landmarks"

## The Met API

https://metmuseum.github.io/

https://github.com/metmuseum/openaccess

**Please limit request rate to 80 requests per second.**

### Painting Objects

- $15\text{,}178$ on website
- $15\text{,}050$ (±100) available in API
- $14\text{,}233$ (±50) have images (according to `hasImages=true` API query param)
- $11\text{,}275$ actually have images that can be downloaded
- $5\text{,}700$ have faces
- $5\text{,}100$ have extractable eye locations/shapes

In [ ]:
obj_ids = PaintingsUtils.get_object_ids()
len(obj_ids)

### Get Object Metadata

In [ ]:
mPU = PaintingsUtils(JSON_DIR, IMG_DIR)

for cnt,oid in enumerate(obj_ids):
  if cnt % 16 == 0:
    print(f"{cnt} / {len(obj_ids)}")

  mPU.get_obj_data(oid)

### Get Faces, Landmarks and Eyes

In [ ]:
mPU = PaintingsUtils(JSON_DIR, IMG_DIR)
mPU.init_face_detector()
mPU.init_face_landmarker()

no_imgs = mPU.no_imgs

last_req = timestamp()

for cnt,oid in enumerate(obj_ids[:10240]):
  if cnt % 16 == 0:
    print(f"{cnt} / {len(obj_ids)}")

  json_landmark_path = f"{JSON_LANDMARKS_DIR}/{oid}.json"
  if path.isfile(json_landmark_path) or oid in no_imgs:
    continue

  obj_data = mPU.get_obj_data(oid)

  if obj_data is None:
    no_imgs.append(oid)
    with open(f"{JSON_DIR}/no_imgs.json", "w") as ofp:
      json.dump(no_imgs, ofp)
    continue

  tdiff = timestamp() - last_req
  if tdiff < 0.75:
    sleep(0.75 - tdiff)

  img_url = obj_data["primaryImage"]
  img_response = requests.get(img_url, stream=True)
  last_req = timestamp()

  if img_response.status_code > 399 or img_response.status_code < 200:
    continue

  img = PImage.open(img_response.raw)

  face_data = mPU.get_face_data(obj_data, img)
  landmark_data = mPU.get_landmark_data(face_data, img)

### Combine Landmark Metadata and Crop Eyes

In [ ]:
import json
import requests

from os import makedirs, path
from PIL import Image as PImage
from time import sleep, time as timestamp

from utils import get_combined_jsons
from utils_paintings import PaintingsUtils

DATA_DIR = "./data"
JSON_DIR = f"{DATA_DIR}/json"
IMG_DIR = f"{DATA_DIR}/image"

JSON_OBJECTS_DIR = f"{JSON_DIR}/objects"
JSON_FACES_DIR = f"{JSON_DIR}/faces"
JSON_LANDMARKS_DIR = f"{JSON_DIR}/landmarks"

In [ ]:
with open(f"{JSON_DIR}/no_imgs.json", "r") as ifp:
  no_imgs = json.load(ifp)

objects_data = get_combined_jsons(JSON_OBJECTS_DIR)
faces_data = get_combined_jsons(JSON_FACES_DIR)

print(len(no_imgs), "have no image")
print(len(objects_data), "have images")
print(len(faces_data), "have faces")

In [ ]:
def has_mp(obj):
  return obj["faces"]["mp"]["count"] > 0

landmarks_data = get_combined_jsons(JSON_LANDMARKS_DIR, has_mp)
len(landmarks_data)

In [ ]:
mPU = PaintingsUtils(JSON_DIR, IMG_DIR)

last_req = timestamp()

for cnt,obj_data in enumerate(landmarks_data[:2]):
  if cnt % 16 == 0:
    print(f"{cnt} / {len(landmarks_data)}")

  img_url = obj_data["primaryImage"]

  tdiff = timestamp() - last_req
  if tdiff < 0.75:
    sleep(0.75 - tdiff)

  img_response = requests.get(img_url, stream=True)
  last_req = timestamp()

  if img_response.status_code > 399 or img_response.status_code < 200:
    continue

  img = PImage.open(img_response.raw)
  mPU.get_eye_images(obj_data, img)